In [3]:
# %% [code]
!pip install git+https://github.com/huggingface/transformers accelerate

  Cloning https://github.com/huggingface/transformers to c:\users\aades\appdata\local\temp\pip-req-build-0bq6536a
  Resolved https://github.com/huggingface/transformers to commit 98d39824ed30e684e5122d04a2d9564efffc4965
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers 'C:\Users\aades\AppData\Local\Temp\pip-req-build-0bq6536a'

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import getpass
import os
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    hf_token = getpass.getpass("Enter your Hugging Face token: ")

if not hf_token:
    raise RuntimeError("A Hugging Face token is required.")

login(token=hf_token)
print("Hugging Face authentication completed.")

In [ ]:
import torch
from transformers import AutoModel, AutoProcessor, AutoImageProcessor
from PIL import Image
import pandas as pd
from tqdm import tqdm
import requests
from io import BytesIO
import numpy as np
import re
import torch.nn.functional as F


MODEL_ID = "facebook/dinov3-vith16plus-pretrain-lvd1689m"
SAVE_PATH = "./dinov3-vith16plus-pretrain-lvd1689m"
DATA_PATH = "/kaggle/input/amlc2025/student_resource/dataset/test.csv"
BATCH_SIZE = 128
DEBUG = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SHARD_NUMBER = 10 # Change this for each Kaggle account
TOTAL_SHARDS = 10 # Set this to the total number of accounts/shards

model = AutoModel.from_pretrained(MODEL_ID, device_map="auto").eval()
processor = AutoImageProcessor.from_pretrained(MODEL_ID)

total_df = pd.read_csv(DATA_PATH)
total_rows = len(total_df)
rows_per_shard = total_rows // TOTAL_SHARDS
start_idx = (SHARD_NUMBER - 1) * rows_per_shard
end_idx = start_idx + rows_per_shard if SHARD_NUMBER < TOTAL_SHARDS else total_rows
df = total_df.iloc[start_idx:end_idx].reset_index(drop=True)
print(f"Processing shard {SHARD_NUMBER}/{TOTAL_SHARDS}: {len(df)} rows")

lim = len(df)
if DEBUG:
    lim = 200

all_image_embeddings = []
all_ids = []

for start_idx in tqdm(range(0, lim, BATCH_SIZE)):
    batch_df = df.iloc[start_idx : min(start_idx + BATCH_SIZE, lim)]
    
    images_to_process = []
    
    for _, row in batch_df.iterrows():
        
        # Process image
        try:
            image_url = row["image_link"]
            if not isinstance(image_url, str) or not (image_url.startswith("http://") or image_url.startswith("https://")):
                raise ValueError("Invalid image URL")

            image_response = requests.get(image_url, stream=True)
            image_response.raise_for_status()
            image = Image.open(image_response.raw).convert("RGB")
            images_to_process.append(image)

        except Exception as e:
            print(f"Failed to process image {row.get('image_link', 'N/A')}. Using a black dummy image instead. Error: {e}")
            image = Image.new('RGB', (224, 224), color='black')
            images_to_process.append(image)

    # Use the processor for images
    # inputs = processor(
    #     text=batch_texts, 
    #     images=images_to_process, 
    #     return_tensors="pt", 
    #     padding="max_length", # Recommended for SigLIP 2
    #     truncation=True,
    #     max_length=64       # Recommended for SigLIP 2
    # ).to(model.device)

    inputs = processor(
        images=images_to_process, 
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        # Get both image and text embeddings from a single model call
        outputs = model(**inputs)
        image_embeddings = outputs.pooler_output
        
        # Normalize the embeddings
        image_embeddings = F.normalize(image_embeddings, p=2, dim=-1)

    for i, (_, row) in enumerate(batch_df.iterrows()):
        all_image_embeddings.append(image_embeddings[i].cpu().numpy())
        all_ids.append(row["sample_id"])
            
    del inputs, outputs, image_embeddings, images_to_process

# Convert lists of embeddings to 2D numpy arrays
all_image_embeddings = np.stack(all_image_embeddings)
all_ids = np.array(all_ids)

# Save image embeddings, text embeddings, and IDs to separate files
np.save(f"image_embeddings_{SHARD_NUMBER}.npy", all_image_embeddings)
np.save(f"sample_ids_{SHARD_NUMBER}.npy", all_ids)

print("Saved image embeddings shape:", all_image_embeddings.shape)
print("Saved sample IDs shape:", all_ids.shape)

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.36G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

Processing shard 10/10: 7500 rows


100%|██████████| 59/59 [39:06<00:00, 39.78s/it]

Saved image embeddings shape: (7500, 1280)
Saved sample IDs shape: (7500,)


In [ ]:
# del model, processor